# 03 - Clean and Process Data

## Objective

Transform the audited raw datasets into analysis-ready tables while preserving the original raw data.

The processing in this notebook will:

* standardize data types and timestamps;
* create reusable lifecycle and business-rule fields;
* handle known data-quality anomalies explicitly;
* preserve incomplete captain journeys rather than dropping them;
* create analysis-ready captain-level and airport-level datasets for downstream analysis.

No raw CSV files will be modified.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
captains = pd.read_csv("../data/captains.csv")
doc_events = pd.read_csv("../data/doc_events.csv")
approvals = pd.read_csv("../data/approvals.csv")
activation = pd.read_csv("../data/activation.csv")
nudges = pd.read_csv("../data/nudges.csv")
airport_hourly = pd.read_csv("../data/airport_hourly.csv")
airport_trips = pd.read_csv("../data/airport_trips.csv")

In [3]:
captains["signup_ts"] = pd.to_datetime(captains["signup_ts"])

doc_events["event_ts"] = pd.to_datetime(doc_events["event_ts"])

approvals["decision_ts"] = pd.to_datetime(approvals["decision_ts"])

activation["first_order_ts"] = pd.to_datetime(
    activation["first_order_ts"]
)

nudges["sent_ts"] = pd.to_datetime(nudges["sent_ts"])

airport_hourly["hour_ts"] = pd.to_datetime(
    airport_hourly["hour_ts"]
)

airport_trips["request_ts"] = pd.to_datetime(
    airport_trips["request_ts"]
)

In [4]:
timestamp_checks = {
    "captains.signup_ts": captains["signup_ts"].dtype,
    "doc_events.event_ts": doc_events["event_ts"].dtype,
    "approvals.decision_ts": approvals["decision_ts"].dtype,
    "activation.first_order_ts": activation["first_order_ts"].dtype,
    "nudges.sent_ts": nudges["sent_ts"].dtype,
    "airport_hourly.hour_ts": airport_hourly["hour_ts"].dtype,
    "airport_trips.request_ts": airport_trips["request_ts"].dtype,
}

timestamp_checks

{'captains.signup_ts': dtype('<M8[ns]'),
 'doc_events.event_ts': dtype('<M8[ns]'),
 'approvals.decision_ts': dtype('<M8[ns]'),
 'activation.first_order_ts': dtype('<M8[ns]'),
 'nudges.sent_ts': dtype('<M8[ns]'),
 'airport_hourly.hour_ts': dtype('<M8[ns]'),
 'airport_trips.request_ts': dtype('<M8[ns]')}

## 2. Define the Observation Window

The dataset is extracted through 2026-06-30 23:59:59. Captains who signed up shortly before the extraction cutoff may not have had sufficient time to complete onboarding or take a first order.

To avoid treating right-censored captains as genuine funnel failures, downstream funnel analysis will use an explicit observation window.

For the primary onboarding funnel, I will require a minimum of 14 days from signup to the data cutoff. This gives each captain a comparable opportunity to progress through onboarding.

The raw data will remain unchanged; the observation eligibility flag will be created as an analytical field.


In [5]:
DATA_CUTOFF = pd.Timestamp("2026-06-30 23:59:59")

captains["days_observed"] = (
    DATA_CUTOFF - captains["signup_ts"]
).dt.total_seconds() / (24 * 60 * 60)

captains["eligible_for_funnel"] = (
    captains["days_observed"] >= 14
)

captains["eligible_for_funnel"].value_counts()

eligible_for_funnel
True     22757
False     2243
Name: count, dtype: int64

In [6]:
captains["days_observed"].describe()

count    25000.000000
mean        85.546792
std         52.144111
min          0.085405
25%         39.534190
50%         83.342002
75%        130.130891
max        180.744433
Name: days_observed, dtype: float64

In [7]:
print(
    "Eligible captains:",
    captains["eligible_for_funnel"].sum()
)

print(
    "Censored captains:",
    (~captains["eligible_for_funnel"]).sum()
)

Eligible captains: 22757
Censored captains: 2243


## 3. Process Document Events

`doc_events` is an event-level table with multiple records per captain and document. For downstream funnel analysis, these events need to be transformed into document-level outcomes before joining them to the captain-level dataset.

For each captain-document combination, I will derive:

* whether the document was uploaded successfully;
* whether it was ever verified successfully;
* whether it was ever rejected;
* the number of attempts;
* the timestamp of the first successful verification.

This preserves the event-level raw data while creating reusable analytical features.


In [8]:
doc_summary = (
    doc_events
    .groupby(["captain_id", "doc_type"])
    .agg(
        attempts=("attempt_no", "max"),
        upload_success=("event_type", lambda x: (x == "upload_success").any()),
        verification_pass=("event_type", lambda x: (x == "verification_pass").any()),
        verification_fail=("event_type", lambda x: (x == "verification_fail").any())
    )
    .reset_index()
)

In [9]:
first_pass = (
    doc_events.loc[
        doc_events["event_type"] == "verification_pass",
        ["captain_id", "doc_type", "event_ts"]
    ]
    .groupby(["captain_id", "doc_type"])["event_ts"]
    .min()
    .reset_index(name="first_pass_ts")
)

In [10]:
doc_summary = doc_summary.merge(
    first_pass,
    on=["captain_id", "doc_type"],
    how="left"
)

In [11]:
doc_summary.head()

,captain_id,doc_type,attempts,upload_success,verification_pass,verification_fail,first_pass_ts
0,CPT100000,AADHAAR,1,True,True,False,2026-05-04 12:16:11.389574146
1,CPT100000,DL,1,True,True,False,2026-05-02 22:37:15.678999608
2,CPT100000,FITNESS,2,True,False,True,NaT
3,CPT100000,PERMIT,1,True,True,False,2026-05-05 14:52:12.185964306
4,CPT100000,RC,1,True,True,False,2026-05-03 17:48:59.486740931


In [12]:
doc_summary.shape

(81173, 7)

## 4. Create Captain-Level Document Features

The onboarding funnel is evaluated at the captain level, so the document-level outcomes will be reshaped into one row per captain.

A document is considered cleared when at least one verification attempt reaches `verification_pass`.

Permit applicability will be handled using the captain's vehicle type: Permit is required for Auto and Cab, but not for ERickshaw.


In [13]:
doc_cleared = (
    doc_summary
    .pivot(
        index="captain_id",
        columns="doc_type",
        values="verification_pass"
    )
    .reset_index()
)

doc_cleared.columns.name = None

In [14]:
doc_cleared = doc_cleared.rename(
    columns={
        "DL": "DL_cleared",
        "RC": "RC_cleared",
        "AADHAAR": "AADHAAR_cleared",
        "PERMIT": "PERMIT_cleared",
        "FITNESS": "FITNESS_cleared",
        "INSURANCE": "INSURANCE_cleared"
    }
)

In [15]:
doc_columns = [
    "DL_cleared",
    "RC_cleared",
    "AADHAAR_cleared",
    "PERMIT_cleared",
    "FITNESS_cleared",
    "INSURANCE_cleared"
]

In [16]:
doc_cleared[doc_columns] = (
    doc_cleared[doc_columns]
    .astype("boolean")
    .fillna(False)
)

In [17]:
doc_cleared.head()

,captain_id,AADHAAR_cleared,DL_cleared,FITNESS_cleared,INSURANCE_cleared,PERMIT_cleared,RC_cleared
0,CPT100000,True,True,False,False,True,True
1,CPT100001,False,True,False,False,False,False
2,CPT100002,True,True,True,True,True,True
3,CPT100003,False,True,False,False,False,False
4,CPT100004,True,True,True,False,False,True


In [18]:
doc_cleared.shape

(23303, 7)

## 5. Preserve the Captain Universe

`captains` is the signup-level base table and therefore defines the complete captain population.

The document summary contains only captains who generated at least one document event. A left join will be used so that captains with no document events remain in the analytical dataset.

For document-stage indicators, missing values after the join will represent no observed successful verification and will be treated as `False`.

This distinction is important because absence of a document event is itself meaningful for the onboarding funnel.


In [19]:
captain_base = captains.merge(
    doc_cleared,
    on="captain_id",
    how="left"
)

In [20]:
captain_base[doc_columns] = (
    captain_base[doc_columns]
    .astype("boolean")
    .fillna(False)
)

In [21]:
print("Captain base rows:", len(captain_base))
print("Unique captains:", captain_base["captain_id"].nunique())

Captain base rows: 25000
Unique captains: 25000


In [22]:
captain_base[doc_columns].sum()

DL_cleared           21954
RC_cleared           15852
AADHAAR_cleared      14095
PERMIT_cleared        8468
FITNESS_cleared       8241
INSURANCE_cleared     4664
dtype: Int32

## 6. Add Approval Outcomes

`approvals` contains the terminal onboarding outcome for each captain, including final status, decision timestamp, last stage reached, and the number of documents cleared.

These fields will be joined at the captain level using `captain_id`.

The approval table is already one row per captain, so this join should preserve the captain-level grain without creating duplicate rows.


In [23]:
approval_columns = [
    "captain_id",
    "decision_ts",
    "final_status",
    "last_stage_reached",
    "docs_cleared"
]

In [24]:
captain_base = captain_base.merge(
    approvals[approval_columns],
    on="captain_id",
    how="left",
    validate="one_to_one"
)

In [25]:
print("Captain base rows:", len(captain_base))
print("Unique captains:", captain_base["captain_id"].nunique())

Captain base rows: 25000
Unique captains: 25000


In [26]:
captain_base["final_status"].value_counts(dropna=False)

final_status
dropped_in_docs    19062
approved            4206
in_progress         1297
rejected             435
Name: count, dtype: int64

## 7. Add Activation Outcomes

`activation` contains post-approval activity for approved captains, including the timestamp of their first completed order and subsequent activity measures.

The table contains only approved captains, so it will be left-joined to the full captain-level dataset. This preserves the complete signup population while leaving activation fields missing for captains who were not approved or have no corresponding activation record.

`first_order_ts` will be used later to distinguish approval from actual activation.


In [27]:
activation_columns = [
    "captain_id",
    "first_order_ts",
    "orders_d7",
    "orders_d30",
    "online_hours_d30"
]

In [28]:
captain_base = captain_base.merge(
    activation[activation_columns],
    on="captain_id",
    how="left",
    validate="one_to_one"
)

In [29]:
print("Captain base rows:", len(captain_base))
print("Unique captains:", captain_base["captain_id"].nunique())

Captain base rows: 25000
Unique captains: 25000


In [30]:
print(
    "Captains with first order:",
    captain_base["first_order_ts"].notna().sum()
)

Captains with first order: 4104


In [31]:
print(
    "Captains without first order:",
    captain_base["first_order_ts"].isna().sum()
)

Captains without first order: 20896


## 8. Derive Onboarding and Activation Durations

Several downstream analyses require the elapsed time between key onboarding milestones.

The following durations will be derived at the captain level:

* `signup_to_approval_days`: time from signup to final approval decision.
* `approval_to_first_order_days`: time from approval to first completed order.

These fields will be calculated only when both timestamps required for the calculation are present. Missing timestamps will therefore remain missing rather than being assigned an arbitrary value.


In [32]:
captain_base["signup_to_approval_days"] = (
    captain_base["decision_ts"] - captain_base["signup_ts"]
).dt.total_seconds() / (24 * 60 * 60)

In [33]:
captain_base["approval_to_first_order_days"] = (
    captain_base["first_order_ts"] - captain_base["decision_ts"]
).dt.total_seconds() / (24 * 60 * 60)

In [34]:
captain_base[
    [
        "signup_to_approval_days",
        "approval_to_first_order_days"
    ]
].describe()

,signup_to_approval_days,approval_to_first_order_days
count,4641.000000,4104.000000
mean,6.429790,1.714744
std,1.547602,1.365636
min,2.542293,0.001303
25%,5.341806,0.721890
50%,6.313038,1.374499
75%,7.387370,2.364522
max,14.109059,11.068693


In [35]:
print(
    "Negative signup-to-approval durations:",
    (captain_base["signup_to_approval_days"] < 0).sum()
)

Negative signup-to-approval durations: 0


In [36]:
print(
    "Negative approval-to-first-order durations:",
    (captain_base["approval_to_first_order_days"] < 0).sum()
)

Negative approval-to-first-order durations: 0


In [37]:
captain_base.shape

(25000, 27)

In [38]:
captain_base.columns.tolist()

['captain_id',
 'signup_ts',
 'city',
 'vehicle_type',
 'acquisition_channel',
 'signup_zone_id',
 'device_tier',
 'app_language',
 'age_band',
 'days_observed',
 'eligible_for_funnel',
 'AADHAAR_cleared',
 'DL_cleared',
 'FITNESS_cleared',
 'INSURANCE_cleared',
 'PERMIT_cleared',
 'RC_cleared',
 'decision_ts',
 'final_status',
 'last_stage_reached',
 'docs_cleared',
 'first_order_ts',
 'orders_d7',
 'orders_d30',
 'online_hours_d30',
 'signup_to_approval_days',
 'approval_to_first_order_days']

In [39]:
print("Rows:", len(captain_base))
print("Unique captain IDs:", captain_base["captain_id"].nunique())
print("Duplicate captain IDs:", captain_base["captain_id"].duplicated().sum())

Rows: 25000
Unique captain IDs: 25000
Duplicate captain IDs: 0


In [40]:
captain_base.isna().sum()

captain_id                          0
signup_ts                           0
city                                0
vehicle_type                        0
acquisition_channel                 0
signup_zone_id                      0
device_tier                         0
app_language                        0
age_band                            0
days_observed                       0
eligible_for_funnel                 0
AADHAAR_cleared                     0
DL_cleared                          0
FITNESS_cleared                     0
INSURANCE_cleared                   0
PERMIT_cleared                      0
RC_cleared                          0
decision_ts                     20359
final_status                        0
last_stage_reached               4393
docs_cleared                        0
first_order_ts                  20896
orders_d7                       20981
orders_d30                      21628
online_hours_d30                21628
signup_to_approval_days         20359
approval_to_